# 예제 01. 합성곱 연산의 직관
빅데이터프로그래밍 · 8주차

## 목표
- 작은 필터가 이미지를 옮겨 다니며 계산하는 과정을 직접 본다
- 필터가 수평선·수직선·모서리를 찾는 것을 눈으로 확인한다
- 특징 맵의 크기가 어떻게 결정되는지 안다

7주차에서 이미지를 784개 숫자로 펼치며 잃었던 이웃 관계를, 여기서 되찾습니다.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

torch.manual_seed(0)


## 1. 손으로 계산해 보기
5×5 이미지에 3×3 필터를 씌웁니다. 필터를 한 칸씩 옮기며 겹치는 부분끼리 곱해 더합니다.


In [ ]:
img = torch.tensor([
    [0., 0., 1., 1., 0.],
    [0., 1., 1., 0., 0.],
    [1., 1., 0., 0., 0.],
    [0., 1., 1., 0., 0.],
    [0., 0., 1., 1., 0.],
])

kernel = torch.tensor([
    [1., 0., -1.],
    [1., 0., -1.],
    [1., 0., -1.],
])           # 세로 방향 변화를 찾는 필터

print("이미지 5x5:"); print(img)
print("\n필터 3x3:"); print(kernel)


In [ ]:
# 왼쪽 위 한 칸만 손으로
patch = img[0:3, 0:3]
print("겹치는 부분:"); print(patch)
print("\n곱해서 더한 값:", (patch * kernel).sum().item())


In [ ]:
# 모든 위치를 돌며 계산 — 이것이 합성곱입니다
out = torch.zeros(3, 3)
for i in range(3):
    for j in range(3):
        out[i, j] = (img[i:i+3, j:j+3] * kernel).sum()

print("특징 맵 3x3:"); print(out)


5×5 이미지에 3×3 필터를 씌우면 3×3이 됩니다. **5 − 3 + 1 = 3** 입니다.


In [ ]:
# PyTorch로 같은 계산 — (batch, 채널, 높이, 너비) 4차원으로 넣습니다
x = img.reshape(1, 1, 5, 5)
w = kernel.reshape(1, 1, 3, 3)

print(F.conv2d(x, w).squeeze())
print("\n손 계산과 같은가:", torch.allclose(F.conv2d(x, w).squeeze(), out))


## 2. 특징 맵 크기 공식

```
출력 크기 = (입력 − 필터 + 2·padding) / stride + 1
```


In [ ]:
import pandas as pd

rows = []
for k, p, s in [(3,0,1), (3,1,1), (5,0,1), (5,2,1), (3,1,2)]:
    out_size = (28 - k + 2*p) // s + 1
    rows.append({"입력": 28, "필터": k, "padding": p, "stride": s, "출력": out_size})
pd.DataFrame(rows)


In [ ]:
# padding=1 을 주면 28이 그대로 유지됩니다 — 자주 쓰는 설정
x28 = torch.randn(1, 1, 28, 28)
print("padding 0:", F.conv2d(x28, torch.randn(1,1,3,3)).shape)
print("padding 1:", F.conv2d(x28, torch.randn(1,1,3,3), padding=1).shape)


## 3. 필터가 찾는 특징
필터의 숫자를 바꾸면 찾는 모양이 달라집니다. 실제 CNN은 이 숫자를 **학습으로** 알아냅니다.


In [ ]:
# 시험용 이미지 — 십자 모양
test = torch.zeros(1, 1, 20, 20)
test[0, 0, 8:12, :] = 1.0      # 가로줄
test[0, 0, :, 8:12] = 1.0      # 세로줄

filters = {
    "수평선 찾기": torch.tensor([[ 1.,  1.,  1.],
                              [ 0.,  0.,  0.],
                              [-1., -1., -1.]]),
    "수직선 찾기": torch.tensor([[ 1., 0., -1.],
                              [ 1., 0., -1.],
                              [ 1., 0., -1.]]),
    "모서리 찾기": torch.tensor([[-1., -1., -1.],
                              [-1.,  8., -1.],
                              [-1., -1., -1.]]),
    "흐리게":     torch.ones(3, 3) / 9,
}

fig, axes = plt.subplots(1, 5, figsize=(16, 3.2))
axes[0].imshow(test.squeeze(), cmap="gray"); axes[0].set_title("원본"); axes[0].axis("off")
for ax, (name, f) in zip(axes[1:], filters.items()):
    res = F.conv2d(test, f.reshape(1,1,3,3), padding=1)
    ax.imshow(res.squeeze(), cmap="gray"); ax.set_title(name); ax.axis("off")
plt.tight_layout(); plt.show()


수평선 필터는 가로줄의 위아래 경계에서, 수직선 필터는 세로줄의 좌우 경계에서 크게 반응합니다.


## 4. 실제 이미지에 씌워 보기


In [ ]:
from torchvision import datasets, transforms

fm = datasets.FashionMNIST("./data", train=True, download=True,
                           transform=transforms.ToTensor())
img0, label0 = fm[0]
x0 = img0.unsqueeze(0)

fig, axes = plt.subplots(1, 5, figsize=(16, 3.2))
axes[0].imshow(img0.squeeze(), cmap="gray"); axes[0].set_title("원본"); axes[0].axis("off")
for ax, (name, f) in zip(axes[1:], filters.items()):
    res = F.conv2d(x0, f.reshape(1,1,3,3), padding=1)
    ax.imshow(res.squeeze(), cmap="gray"); ax.set_title(name); ax.axis("off")
plt.tight_layout(); plt.show()


## 5. 필터 하나 vs 여러 개
`out_channels` 는 서로 다른 필터를 몇 개 쓸지입니다. 필터마다 특징 맵이 하나씩 나옵니다.


In [ ]:
conv = nn.Conv2d(in_channels=1, out_channels=8, kernel_size=3, padding=1)

print("가중치 shape:", conv.weight.shape)     # (8, 1, 3, 3) — 필터 8개
print("편향 shape  :", conv.bias.shape)
print("파라미터    :", sum(p.numel() for p in conv.parameters()), "= 8*1*3*3 + 8")

out = conv(x0)
print("\n입력:", tuple(x0.shape), "→ 출력:", tuple(out.shape))


In [ ]:
# 학습 전 무작위 필터 8개의 결과 — 아직 의미 있는 특징은 아닙니다
fig, axes = plt.subplots(1, 8, figsize=(16, 2.4))
for i, ax in enumerate(axes):
    ax.imshow(out[0, i].detach(), cmap="gray"); ax.set_title(f"filter {i}", fontsize=10); ax.axis("off")
plt.tight_layout(); plt.show()


## 직접 해보기
1. 5×5 필터로 바꾸면 특징 맵 크기가 얼마가 되나요? 계산하고 코드로 확인하세요.
2. 위 `filters` 에 자기가 만든 필터를 하나 추가해 결과를 보세요.
3. `stride=2` 를 주면 출력이 얼마가 되나요?


In [ ]:
# 여기에 작성하세요
